## Desarrollo dataset letras 

In [12]:
# imports
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode, split_transpose)
from numpy.random import RandomState


In [ ]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()


In [16]:
# Mostrar la "cabecera" de X_letter
pd.DataFrame(X_letter).head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,2,4,4,3,2,7,8,2,9,11,7,7,1,8,5,6
1,4,7,5,5,5,5,9,6,4,8,7,9,2,9,7,10
2,7,10,8,7,4,8,8,5,10,11,2,8,2,5,5,10
3,4,9,5,7,4,7,7,13,1,7,6,8,3,8,0,8
4,6,7,8,5,4,7,6,3,7,10,7,9,3,8,3,7
5,4,7,5,5,3,4,12,2,5,13,7,5,1,10,1,7
6,6,10,8,8,4,7,8,2,5,10,7,8,5,8,1,8
7,1,0,2,0,1,6,10,7,2,7,5,8,2,7,4,9
8,5,9,7,6,7,7,7,2,4,9,8,9,7,6,2,8
9,1,0,2,1,1,5,7,8,6,7,6,6,2,8,3,8


In [15]:
# Mostrar la "cabecera" de y_letter
pd.DataFrame(y_letter).head(10)

,0
0,Z
1,P
2,S
3,H
4,H
5,F
6,N
7,R
8,M
9,D


In [ ]:
# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1)) # hago reshape para que quede como matriz columna

In [17]:
# Mostrar la "cabecera" de y_letter_encoded
pd.DataFrame(y_letter_encoded).head(10)

,0
0,25
1,15
2,18
3,7
4,7
5,5
6,13
7,17
8,12
9,3


In [18]:
# shapes
X_letter.shape, y_letter_encoded.shape

((20000, 16), (20000, 1))

In [7]:
# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


In [19]:
qda = QDA()

qda.fit(X_letter.T, y_letter_encoded)

qda.predict(X_letter.T[:, :5])

array([[25,  5, 18,  7,  7]])

### HASTA ACA

Dataset cargado ✅
Labels encoded ✅
Benchmark configurado ✅
QDA entrenó  ✅
La predicción devolvió clases válidas ✅

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

## Respuesta

QDA
inv_cov # (p,p)
unbiased_x # (p,1)
return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x
(1,p) @ (p,p) @ (p,1) → (1,1) → escalar.

TensorizedQDA
tensor_inv_covs # (k, p, p)
tensor_means # (k, p, 1)

unbiased_x: (k,p,1)
unbiased_x.transpose(0,2,1) → (k,1,p)
tensor_inv_cov: (k,p,p)

(k,1,p) @ (k,p,p) → hace producto matriz-matriz por clase.
Resultado: (k,1,p)

(k,1,p) @ unbiased_x (k,p,1) → (k,1,1)

inner_prod tiene shape (k, 1, 1).

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

## Respuesta

### Objetivo
Poder eliminar el ciclo for de predict. Es decir, predecir las n observaciones y k clases en un mismo paso. 

En BaseBayesianClassifier

predict(self, X) → tiene ciclo for recorre las n observaciones, y llama a predict_one, donde TensorizedQDA ya paraleliza las k clases. 

-------------
def predict(self, X): → llama a predict_one por cada observación

def _predict_one(self, x): → llama a _predict_log_conditional para hacer el argmax de la suma

$$
\log\hat{f}_j(x) + \log\hat{\pi}_j
$$

def _predict_log_conditional(self, x, class_idx)
$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$
----------------

Buscamos calcular la forma cuadrática para muchas observaciones a la vez:

$$(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$$

Donde usamos : _predict_log_conditional(x, class_idx)

unbiased_x = x - mean_j

unbiased_x.T @ inv_cov_j @ unbiased_x



Definir una clase FasterQDA que herede de TensorizedQDA y redefina predict para predecir todas las observaciones juntas, sin el for sobre filas de X



In [22]:
class FasterQDA(TensorizedQDA):
    """
    Versión vectorizada (sin bucles en predict), 
    con matriz intermedia ineficiente (matriz N x N).
    """

    def _predict_log_conditionals_batch(self, X):
        # Dimensiones iniciales
        k, p, _ = self.tensor_means.shape # self.tensor_means: (k, p, 1)    → k clases, p features
        n = X.shape[1]                    # X: (p, n)                       → p features, n observaciones

        # Cálculo resta: D = X - medias
        D = X[None, :, :] - self.tensor_means   # X: (1, p, n) - Medias: (k, p, 1) → D: (k, p, n)

        # Cálculo de la distancia cuadrática 
        quad_terms = np.empty((k, n)) # Almacena distancias por cada k clase y observación n

        for j in range(k):
            D_j = D[j]                          # Shape: (p, n)
            inv_cov_j = self.tensor_inv_cov[j]  # Shape: (p, p)

            # ---------------------------------------------------------
            # Generación de matriz n x n
            # ---------------------------------------------------------
            # Al multiplicar D_j.T (n, p) @ inv (p, p) @ D_j (p, n), el resultado final es una matriz de (n, n)
            # solo nos importa cada dato consigo mismo (la diagonal)
            
            Q_j = D_j.T @ inv_cov_j @ D_j      # matriz (n, n)

            # Nos quedamos con la diagonal 
            quad_terms[j, :] = np.diag(Q_j)    
            # ---------------------------------------------------------

        
        # log_det es (k,). Lo convertimos a (k, 1) para operar con quad_terms
        log_det = np.log(LA.det(self.tensor_inv_cov))[:, None]
        
        # Fórmula final: 0.5 * log_det - 0.5 * distancia
        return 0.5 * log_det - 0.5 * quad_terms

    def predict(self, X):
        # traigo el log-condicionales 
        log_cond = self._predict_log_conditionals_batch(X)

        # Sumamos el log_a_priori 
        log_post = self.log_a_priori[:, None] + log_cond

        # Elegimos la clase ganadora → por columna
        y_hat = np.argmax(log_post, axis=0)

        return y_hat.reshape(1, -1)